In [ ]:
key = "여러분들의 Key값"
%env OPENAI_API_KEY = {key}

In [ ]:
import openai
openai.__version__

In [ ]:
import json
import pandas as pd
import urllib.request
from openai import OpenAI

client = OpenAI()

In [ ]:
urllib.request.urlretrieve('https://raw.githubusercontent.com/MrBananaHuman/CounselGPT/main/total_kor_multiturn_counsel_bot.jsonl', 'total_kor_multiturn_counsel_bot.jsonl')

with open('total_kor_multiturn_counsel_bot.jsonl', 'r', encoding='utf-8') as file:
    original_jsonl_data = [json.loads(line) for line in file]
print('데이터의 개수:', len(original_jsonl_data))

In [ ]:
original_jsonl_data = original_jsonl_data[:200]
print('데이터의 개수:', len(original_jsonl_data))

In [ ]:
print(original_jsonl_data[0])

In [ ]:
speaker_dict = {'내담자': 'user', '상담사': 'assistant'}

def transform_to_new_format(original_data, speaker_dict):
    transformed_data = []
    for conversation in original_data:
        current_conversation = {"messages": [{"role": "system", "content": "당신은 정서적으로 심리가 불안한 사용자를 위로해주는 심리 상담 챗봇입니다. 성심성의껏 상담해주세요"}]}
        for item in conversation:
            current_conversation["messages"].append({
                "role": speaker_dict[item["speaker"]],
                "content": item["utterance"]
            })
        if current_conversation['messages'][-1]['role'] == 'user':
            current_conversation['messages'] = current_conversation['messages'][:-1]
        transformed_data.append(current_conversation)

    return transformed_data

result = transform_to_new_format(original_jsonl_data, speaker_dict)
print('데이터의 개수:', len(result))

In [ ]:
print(result[0])

In [ ]:
def save_jsonl_file(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        for item in data:
            json.dump(item, file, ensure_ascii=False)
            file.write('\n')

save_jsonl_file(result, 'messages.jsonl')

In [ ]:
client.files.create(
    file=open("messages.jsonl", "rb"),
    purpose='fine-tune'
)

In [ ]:
client.fine_tuning.jobs.create(
  training_file="file-fqyrfz01AUF6Lk61test", 
  model="gpt-4o-mini-2024-07-18"
)

In [ ]:
finetuning_lst = client.fine_tuning.jobs.list(limit=10)

In [ ]:
for elem in finetuning_lst.data:
    if elem.training_file == "file-fqyrfz01AUF6Lk61test":
        print(elem.status) 

In [ ]:
finetuning_list = client.fine_tuning.jobs.list(limit=10)

for elem in finetuning_list.data:
    if elem.training_file == "file-fqyrfz01AUF6Lk61test":
        print(elem.fine_tuned_model)

In [ ]:
def respone_by_chatgpt(model, messages):
    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    print(completion.choices[0].message.content)

model = 'gpt-4o-mini'
system_prompt = "당신은 정서적으로 심리가 불안한 사용자를 위로해주는 심리 상담 챗봇입니다. 상담하면서 여러 가지 방법을 알려주세요."
user_input = "요즘 혼자인 것 같아서 외로워요"

messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

respone_by_chatgpt(model, messages)

In [ ]:
model = 'ft:gpt-4o-mini-2024-07-18:personal::A0ib9VZf'
respone_by_chatgpt(model, messages)

In [ ]:
user_first_input = "요즘 혼자인 것 같아서 외로워요"
assistant_answer = "혼자 있는 것 같아서 외로움을 느끼신다는 건, 친구나 가족과의 소통 부족으로 인해 자기감정을 충족시키지 못하고 있다는 것을 의미합니다. 내담자님이 외로움을 느끼는 이유에 대해 좀 더 말씀해주실 수 있을까요?"
user_second_input = "저도 뭐가 문제인지 모르겠어요"

messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_first_input},
        {"role": "assistant", "content": assistant_answer},
        {"role": "user", "content": user_second_input}
    ]
model='ft:gpt-4o-mini-2024-07-18:personal::A0ib9VZf'
respone_by_chatgpt(model, messages)

In [ ]:
import gradio as gr

def predict(user_input, history):
    history.append({"role": "user", "content": user_input})

    gpt_response = client.chat.completions.create(
        model=model,
        messages=history
    )

    response = gpt_response.choices[0].message.content
    history.append({"role": "assistant", "content": response})
    messages = [(history[i]["content"], history[i+1]["content"])
                for i in range(1, len(history), 2)]
    
    return messages, history


with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="ChatBot")

    state = gr.State([{
        "role": "system",
        "content": "당신은 정서적으로 심리가 불안한 사용자를 위로해주는 심리 상담 챗봇입니다. 성심성의껏 상담해주세요."
    }])

    with gr.Row():
        txt = gr.Textbox(show_label=False,
                         placeholder="심리 상담 챗봇에게 심리 상담을 받아보세요.")

    txt.submit(predict, [txt, state], [chatbot, state])

demo.launch(debug=True, share=True)